In [1]:
import pandas as pd

## Base vs RAG correctness

In [3]:
for method in ["base", 'rag']:
    df = pd.read_csv(f"res-{method}-judge.csv")
    correct = len(df[df['judge_output'] == 'Yes'])
    total = len(df)
    percentage_correct = (correct * 100)/total
    print(f'{method.upper()}, correct = {correct}/{total} ({percentage_correct:.2f}%)')

BASE, correct = 435/918 (47.39%)
RAG, correct = 736/918 (80.17%)


## Error Analysis

In [15]:
df = pd.read_csv('res-rag-judge.csv')
incorrect = df[df['judge_output'] == 'No']

In [50]:
error_df = incorrect.sample(50, random_state=42)

In [31]:
from utils.chroma import init_chroma_client, init_collection, query_collection
from IPython.display import clear_output

In [6]:
client = init_chroma_client()
collection = init_collection(client)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
errors = []
docs_str = []
rows = list(error_df.iterrows())

for i, item in enumerate(rows):
    clear_output(wait=True)
    row = item[-1]
    query_result = query_collection(collection, row['question'])
    docs = query_result["documents"][0]
    
    print(f"\nROW {i+1} of {len(error_df)}")
    print(f"Question: {row['question']}")
    print(f"Answer: {row['answer']}")
    print(f"Prediction: {row['prediction']}")
    print(f"\nDocs:")
    for j, doc in enumerate(docs):
        print(f"{j+1}. {doc}")
        
    print()
    
    # Get input
    note = input("Error Type (Generation, Retrieval, Annotation, Judgement, Vague question): ")
    errors.append(note)
    docs_str.append(str(docs))

# Save results to csv
output_file = 'error_annotations.csv'
error_df['error_type'] = errors
error_df['docs'] = docs_str
error_df.to_csv(output_file, index=False)


ROW 50 of 50
Question: hard
Answer: hard
Prediction: The provided context is insufficient to answer the question.

Docs:
1. |
2.  ;
3. |-
4. |-
5. |}



Error Type (Generation, Retrieval, Annotation, Judgement, Vague question):  Vague question


In [53]:
error_df['error_type'].value_counts()

error_type
Retrieval         13
Generation        13
Annotation        10
Vague question     9
Judgement          5
Name: count, dtype: int64

## RAG vs RAG-Reranker correctness

In [5]:
for method in ["rag", 'rag-reranked']:
    df = pd.read_csv(f"res-{method}-judge.csv")
    correct = len(df[df['judge_output'] == 'Yes'])
    total = len(df)
    percentage_correct = (correct * 100)/total
    print(f'{method.upper()}, correct = {correct}/{total} ({percentage_correct:.2f}%)')

RAG, correct = 736/918 (80.17%)
RAG-RERANKED, correct = 734/918 (79.96%)


In [6]:
rag_error_df = pd.read_csv('error_annotations.csv')

In [7]:
df = pd.read_csv('res-rag-reranked-judge.csv')
reranked_incorrect_df = df[df['judge_output'] == 'No']

In [8]:
reranked_incorrect_df

,question_id,question,answer,prediction,judge_output
17,34,Why did Lincoln issue the Emancipation Proclam...,To free slaves,Lincoln issued the Emancipation Proclamation t...,No
24,49,What trail did Lincoln use a Farmers' Almanac ...,"he defended William ""Duff"" Armstrong",The provided context does not mention Lincoln ...,No
28,83,Who graduated in ecclesiastical law at the ear...,Amedeo Avogadro,John Adams graduated in ecclesiastical law at ...,No
37,123,Did the scientific community not reserve great...,yes,Yes. The scientific community did not reserve ...,No
39,136,What happened in 1833?,Avogadro had been recalled to Turin university,The context provided does not mention what hap...,No
...,...,...,...,...,...
908,1705,"What defended slavery, owned slaves and S08_se...",His father,The context provided does not mention anyone w...,No
909,1706,Who did Wilson win in 1917?,Irish Americans,Wilson won the 1917 U.S. presidential election.,No
912,1709,What lived in Columbia?,Wilson,The context does not provide information about...,No
914,1711,Did he not cast his ballot for John M. Palmer ...,Yes,"Yes, he cast his ballot for John M. Palmer, th...",No


In [14]:
result = pd.merge(rag_error_df, reranked_incorrect_df, on='question_id', how='inner')

In [15]:
result['error_type'].value_counts()

error_type
Generation        10
Annotation         8
Vague question     8
Retrieval          5
Judgement          3
Name: count, dtype: int64

In [16]:
rag_df = pd.read_csv('res-rag-judge.csv')
reranker_df = pd.read_csv('res-rag-reranked-judge.csv')

In [17]:
rag_df[rag_df['judge_output'] == 'Yes']

,question_id,question,answer,prediction,judge_output
0,0,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, Abraham Lincoln was the sixteenth Preside...",Yes
1,2,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, Lincoln signed the National Banking Act o...",Yes
2,4,Did his mother die of pneumonia?,no,"No, his mother did not die of pneumonia. She d...",Yes
3,6,How many long was Lincoln's formal education?,18 months,Lincoln's formal education consisted of about ...,Yes
4,8,When did Lincoln begin his political career?,1832,Lincoln began his political career in 1832.,Yes
...,...,...,...,...,...
910,1707,Was Wilson awarded the 1919 Nobel Peace Prize?,Yes,"Yes, Wilson was awarded the 1919 Nobel Peace P...",Yes
911,1708,Was Wilson a remarkably effective writer and t...,Yes,"Yes, Wilson was a remarkably effective writer ...",Yes
913,1710,Was Wilson president of the American Political...,Yes,"Yes, according to the context, Wilson was pres...",Yes
915,1712,Did Wilson not spend 1914 through the beginnin...,Yes,"Yes, Wilson tried to keep America out of the w...",Yes


In [18]:
reranker_df[reranker_df['judge_output'] == 'Yes']

,question_id,question,answer,prediction,judge_output
0,0,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, according to the context, Abraham Lincoln...",Yes
1,2,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, Lincoln signed the National Banking Act o...",Yes
2,4,Did his mother die of pneumonia?,no,"No, his mother did not die of pneumonia. She d...",Yes
3,6,How many long was Lincoln's formal education?,18 months,Lincoln's formal education consisted of about ...,Yes
4,8,When did Lincoln begin his political career?,1832,Lincoln began his political career in 1832.,Yes
...,...,...,...,...,...
910,1707,Was Wilson awarded the 1919 Nobel Peace Prize?,Yes,"Yes, Wilson was awarded the 1919 Nobel Peace P...",Yes
911,1708,Was Wilson a remarkably effective writer and t...,Yes,"Yes, based on the context, Wilson was a remark...",Yes
913,1710,Was Wilson president of the American Political...,Yes,"Yes, based on the context provided.",Yes
915,1712,Did Wilson not spend 1914 through the beginnin...,Yes,"Yes, Wilson did try to keep America out of the...",Yes


In [19]:
df2 = pd.merge(rag_df, reranker_df, on='question_id', suffixes=("_rag", "_reranked"))

In [20]:
df2

,question_id,question_rag,answer_rag,prediction_rag,judge_output_rag,question_reranked,answer_reranked,prediction_reranked,judge_output_reranked
0,0,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, Abraham Lincoln was the sixteenth Preside...",Yes,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, according to the context, Abraham Lincoln...",Yes
1,2,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, Lincoln signed the National Banking Act o...",Yes,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, Lincoln signed the National Banking Act o...",Yes
2,4,Did his mother die of pneumonia?,no,"No, his mother did not die of pneumonia. She d...",Yes,Did his mother die of pneumonia?,no,"No, his mother did not die of pneumonia. She d...",Yes
3,6,How many long was Lincoln's formal education?,18 months,Lincoln's formal education consisted of about ...,Yes,How many long was Lincoln's formal education?,18 months,Lincoln's formal education consisted of about ...,Yes
4,8,When did Lincoln begin his political career?,1832,Lincoln began his political career in 1832.,Yes,When did Lincoln begin his political career?,1832,Lincoln began his political career in 1832.,Yes
...,...,...,...,...,...,...,...,...,...
913,1710,Was Wilson president of the American Political...,Yes,"Yes, according to the context, Wilson was pres...",Yes,Was Wilson president of the American Political...,Yes,"Yes, based on the context provided.",Yes
914,1711,Did he not cast his ballot for John M. Palmer ...,Yes,"Yes, he cast his ballot for John M. Palmer, th...",No,Did he not cast his ballot for John M. Palmer ...,Yes,"Yes, he cast his ballot for John M. Palmer, th...",No
915,1712,Did Wilson not spend 1914 through the beginnin...,Yes,"Yes, Wilson tried to keep America out of the w...",Yes,Did Wilson not spend 1914 through the beginnin...,Yes,"Yes, Wilson did try to keep America out of the...",Yes
916,1713,"Was Wilson , a staunch opponent of antisemitis...",Yes,"Yes, Wilson was sympathetic to the plight of J...",Yes,"Was Wilson , a staunch opponent of antisemitis...",Yes,Yes. The evidence states that Wilson was a sta...,Yes


In [24]:
fixed      = df2[(df2.judge_output_rag == "No")  & (df2.judge_output_reranked == "Yes")]
regressed  = df2[(df2.judge_output_rag == "Yes") & (df2.judge_output_reranked == "No")]
both_wrong = df2[(df2.judge_output_rag == "No")  & (df2.judge_output_reranked == "No")]

print(f"fixed={len(fixed)}  regressed={len(regressed)}  net={len(fixed)-len(regressed)}")
print((df2.judge_output_rag == "Yes").sum(), (df2.judge_output_reranked == "Yes").sum())  # should be 714 and 736

fixed=55  regressed=33  net=22
714 736


In [25]:
regressed

,question_id,question_rag,answer_rag,prediction_rag,judge_output_rag,question_reranked,answer_reranked,prediction_reranked,judge_output_reranked
102,235,Did Coolidge get in Amherst College?,Yes,"Yes, Coolidge got into Amherst College. He spe...",Yes,Did Coolidge get in Amherst College?,Yes,"No, Coolidge did not get into Amherst College....",No
103,237,What caused Calvin Jr.'s death?,heart attack,Calvin Jr. died of a heart attack at his home ...,Yes,What caused Calvin Jr.'s death?,heart attack,Calvin Jr. died from an infected blister on hi...,No
143,337,Did Cartier not use the word ` Canada ' to ref...,yes,Yes. Cartier used the word 'Canada' to refer t...,Yes,Did Cartier not use the word ` Canada ' to ref...,yes,Yes. Cartier used the word 'Canada' to refer t...,No
161,369,What does the word duck mean?,It is the common name for a number of species ...,"The word ""duck"" refers to a bird in the Anatid...",Yes,What does the word duck mean?,It is the common name for a number of species ...,"The word ""duck"" means the bird, derived from t...",No
171,389,What expression is part of a conceptual framew...,Quacks like a duck,"The expression ""duck test"" is part of a concep...",Yes,What expression is part of a conceptual framew...,Quacks like a duck,"The expression ""if a bird quacks and looks lik...",No
174,395,Is Egypt the most populated country in Africa?,no,No. Egypt is the second-most populous country ...,Yes,Is Egypt the most populated country in Africa?,no,"Yes, Egypt is the most populated country in Af...",No
177,401,What religions has Egypt outlawed?,"All but Christianity, Islam, and Judaism",Egypt has outlawed all religions and belief ex...,Yes,What religions has Egypt outlawed?,"All but Christianity, Islam, and Judaism",Egypt has outlawed all religions and belief ex...,No
194,434,How many living species of African Elephants a...,2,"The African Elephant genus contains two (or, a...",Yes,How many living species of African Elephants a...,2,The context states there are two (or three) li...,No
263,591,Has another US President been an Eagle Scout?,no,"Yes, Gerald Ford was an Eagle Scout and is the...",Yes,Has another US President been an Eagle Scout?,no,"Yes, Gerald Ford was an Eagle Scout and he is ...",No
264,593,Did Ford need to do extra work to pay for coll...,yes,"Yes, Ford needed to do extra work to pay for c...",Yes,Did Ford need to do extra work to pay for coll...,yes,"No, Ford did not need to do extra work to pay ...",No
